# Урок 5.4. Прямой inference через llama-cpp-python

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 4 из 7

Три урока подряд мы работали с Ollama — демоном, который прячет от нас движок
llama.cpp. Сегодня уберём посредника: загрузим ту же модель **прямо в свой
Python-процесс** и получим то, чего демон не даёт, — полный контроль над
генерацией, включая грамматики, которые физически не позволяют модели
нарушить формат.

## 🎯 Что вы узнаете

- Чем llama-cpp-python отличается от Ollama и когда прямой inference — правильный выбор
- Как использовать уже скачанные Ollama-модели без повторной загрузки (blobs = GGUF)
- Как загрузить GGUF и разговаривать с моделью: create_chat_completion, стриминг, параметры
- Сколько потоков CPU реально помогают — и почему «больше» не значит «быстрее»
- Грамматики GBNF и JSON-схемы: формат, гарантированный на уровне сэмплера, а не «по просьбе»


## Тот же движок, минус демон

Вспомним архитектуру из урока 5.3: Ollama — это демон, который принимает HTTP
и держит runner-процессы с llama.cpp внутри. **llama-cpp-python** — это Python-обвязка
над той же библиотекой llama.cpp, только без демона: модель живёт в памяти
**вашего** процесса, и вызов модели — обычный вызов функции.

```
С Ollama (уроки 5.1-5.3):
  ваш процесс --HTTP--> демон ollama --> runner (llama.cpp + модель в RAM)

С llama-cpp-python (этот урок):
  ваш процесс [llama.cpp + модель в RAM]     <- всё в одном процессе, без сети
```

Аналогия из мира БД — та же, что вела нас весь курс:

| | Ollama | llama-cpp-python |
|---|---|---|
| Аналог | PostgreSQL-сервер | SQLite |
| Кто пользуется | много клиентов по сети | один процесс — владелец |
| Управление моделями | pull/rm, реестр, версии | вы сами следите за GGUF-файлами |
| Очередь, параллелизм, keep_alive | встроены (урок 5.3) | ваша забота |
| Контроль генерации | options из фиксированного списка | всё: грамматики, logit bias, свой сэмплер |
| Деплой | отдельная служба | `pip install`, прилетает с приложением |

Когда прямой inference — правильный выбор:

- **CLI-утилиты и desktop-приложения**: не заставлять пользователя ставить
  и держать демона ради одной функции.
- **Пайплайны и батчи**: скрипт ночной обработки, которому не нужен
  постоянно живущий сервис.
- **Жёсткие гарантии формата**: грамматики GBNF — киллер-фича этого урока.
- **Исследования**: доступ к логитам, перплексии, низкоуровневым параметрам.

Когда он выбор неправильный: несколько клиентов, очереди, горячая замена моделей —
это всё Ollama (урок 5.3) уже умеет, а тут придётся писать самим.


## Установка

Пакет капризный: это C++-библиотека, которой нужно собранное под вашу машину ядро.
Порядок действий (подробности — в README модуля):

1. Сначала пробуем готовое CPU-колесо (у меня для Python 3.13 сработало именно оно,
   поставился llama-cpp-python 0.3.x):

   ```powershell
   pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
   ```

2. Если колеса под вашу версию Python нет — сборка из исходников (Visual Studio
   Build Tools + CMake на Windows, `build-essential cmake` на Linux) или отдельный
   venv с Python 3.12.

Все ячейки ниже аккуратно пропускаются, если пакет не установлен, — но практическая
ценность урока, как обычно, в живом запуске.


## Модель у вас уже есть: достаём GGUF из хранилища Ollama

llama-cpp-python ест файлы GGUF. Обычный путь — скачать с Hugging Face
(`huggingface_hub`, сборки вроде `bartowski/...-GGUF`). Но вспомните урок 5.3:
**модель Ollama — это manifest + blobs, и блоб с весами — это и есть GGUF-файл**.
Ничего скачивать не нужно — найдём его на диске:

1. Манифест лежит в `~/.ollama/models/manifests/registry.ollama.ai/library/<модель>/<тег>`.
2. В нём — список слоёв; нам нужен слой с `mediaType = application/vnd.ollama.image.model`.
3. Его digest — имя файла в `~/.ollama/models/blobs/`.

Два честных предупреждения:

- Блоб — собственность Ollama: **только читаем**, не переименовываем и не двигаем.
  `ollama rm qwen2.5:3b` удалит файл — ваш код перестанет находить модель.
- Для настоящего продакшена на llama-cpp-python скачайте **свой** GGUF-файл
  и положите рядом с приложением — не зависьте от чужого кеша.


In [ ]:
import json
from pathlib import Path

try:
    from llama_cpp import Llama, LlamaGrammar
    HAS_LLAMA_CPP = True
except ImportError:
    HAS_LLAMA_CPP = False
    print("llama-cpp-python не установлен — ячейки с inference будут пропущены")


def find_ollama_gguf(model: str, tag: str) -> Path | None:
    """Путь к GGUF-блобу скачанной Ollama-модели (или None, если её нет)."""
    manifest_path = (Path.home() / ".ollama/models/manifests"
                     / "registry.ollama.ai/library" / model / tag)
    if not manifest_path.exists():
        return None
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    digest = next(layer["digest"] for layer in manifest["layers"]
                  if layer["mediaType"] == "application/vnd.ollama.image.model")
    blob = Path.home() / ".ollama/models/blobs" / digest.replace(":", "-")
    return blob if blob.exists() else None


GGUF_PATH = find_ollama_gguf("qwen2.5", "3b")
if GGUF_PATH:
    with open(GGUF_PATH, "rb") as f:
        magic = f.read(4)
    print(f"нашли: {GGUF_PATH.name[:32]}…  {GGUF_PATH.stat().st_size / 1024**3:.2f} ГиБ, "
          f"магия файла: {magic}")
else:
    print("qwen2.5:3b не найдена в хранилище Ollama (ollama pull qwen2.5:3b)")
    # Запасной путь — скачать GGUF с Hugging Face (~2 ГБ), раскомментируйте:
    # from huggingface_hub import hf_hub_download
    # GGUF_PATH = Path(hf_hub_download("Qwen/Qwen2.5-3B-Instruct-GGUF",
    #                                  "qwen2.5-3b-instruct-q4_k_m.gguf"))

Первые 4 байта — `b'GGUF'`: сигнатура формата. Никакой магии, обычный файл.

## Загружаем модель в свой процесс

Ключевые параметры конструктора `Llama`:

| Параметр | Дефолт | Зачем |
|---|---|---|
| `n_ctx` | **512!** | Размер контекста. Дефолт крошечный — первая же грабля, см. частые ошибки |
| `n_threads` | авто | Потоки CPU для генерации (замерим ниже) |
| `n_threads_batch` | авто | Потоки для prefill |
| `n_gpu_layers` | 0 | Слоёв на GPU; 0 = чистый CPU (для GPU нужно колесо с CUDA) |
| `verbose` | True | Болтливые логи движка в stderr; в ноутбуке выключаем |

Обратите внимание: модель займёт ~2 ГБ **в памяти вашего процесса** — проверьте
диспетчером задач. Это не демон, которому «выдали» память, это ваш собственный RSS.


In [ ]:
import time

LLM = None
if HAS_LLAMA_CPP and GGUF_PATH:
    t0 = time.perf_counter()
    LLM = Llama(model_path=str(GGUF_PATH), n_ctx=4096, verbose=False)
    print(f"модель загружена за {time.perf_counter() - t0:.1f} с")
else:
    print("Пропущено: нужен llama-cpp-python и скачанная модель")

## Разговор: create_chat_completion

Помните урок 1.6, где мы вручную собирали ChatML-разметку для base-модели?
Здесь это не нужно: **шаблон чата хранится в метаданных GGUF**, и
`create_chat_completion` применяет его сам. API намеренно похож на OpenAI:
`messages`, `max_tokens`, `temperature`, в ответе — `choices` и `usage`.


In [ ]:
def chat(messages: list[dict], max_tokens: int = 100, **kwargs) -> str:
    """Один вызов чата с замером скорости (usage + настенные часы)."""
    t0 = time.perf_counter()
    out = LLM.create_chat_completion(messages=messages, max_tokens=max_tokens,
                                     temperature=0, seed=42, **kwargs)
    elapsed = time.perf_counter() - t0
    usage = out["usage"]
    print(f"[{usage['prompt_tokens']} in + {usage['completion_tokens']} out за {elapsed:.1f} с "
          f"≈ {usage['completion_tokens'] / elapsed:.1f} ток/с, "
          f"finish={out['choices'][0]['finish_reason']}]")
    return out["choices"][0]["message"]["content"]


if LLM:
    answer = chat([
        {"role": "system", "content": "Отвечай кратко и по-русски."},
        {"role": "user", "content": "Что такое GIL в Python? Одно предложение."},
    ], max_tokens=60)
    print(answer)
else:
    print("Пропущено")

В моём прогоне: ~10–11 ток/с и ответ с типичной для 3B неточностью («глобальный
индикатор блокировки» — ну почти). Модель та же, что в Ollama, — и её ограничения
те же: прямой inference не добавляет ума, только контроль.

### Тот же движок — какая скорость?

В уроке 5.2 Ollama в CPU-режиме давала **15–17 ток/с**. llama-cpp-python на том же
файле у меня показывает **от 10 до 15 ток/с** в разных прогонах: иногда паритет
с Ollama (в упражнении 3 ниже вышло 15.3 против 15.5), иногда на четверть медленнее.
Разброс объясняют две вещи: тепловой режим ноутбука (тот же троттлинг, что портил
КПД в уроке 5.2) и сборка — готовое колесо собрано «на всех», а Ollama возит
несколько вариантов ядра и на старте выбирает лучший под ваш CPU.

Мораль двойная: движок тот же, магии нет ни в одну сторону; и единичный замер —
мой в том числе — ничего не доказывает. Гоняйте свой бенчмарк из урока 5.2.


## Сколько потоков CPU нужно на самом деле

Урок 5.2 доказал: генерация упирается в пропускную способность RAM. Значит,
у потоков должен быть потолок пользы — полоса памяти одна на всех. Проверим:
прогоним одну и ту же генерацию с разным `n_threads` (каждый раз загружая
модель заново — это займёт минуту-другую).


In [ ]:
if HAS_LLAMA_CPP and GGUF_PATH:
    print(f"{'n_threads':>9} | {'генерация ток/с':>15}")
    print("-" * 29)
    for n_threads in (2, 4, 8, None):
        probe = Llama(model_path=str(GGUF_PATH), n_ctx=2048,
                      n_threads=n_threads, verbose=False)
        t0 = time.perf_counter()
        out = probe.create_completion("Список из 15 названий городов через запятую: ",
                                      max_tokens=40, temperature=0, seed=42)
        elapsed = time.perf_counter() - t0
        tokens = out["usage"]["completion_tokens"]
        print(f"{str(n_threads) if n_threads else 'авто':>9} | {tokens / elapsed:>15.1f}")
        del probe
else:
    print("Пропущено")

Мой прогон: ~10 ток/с на двух потоках, **плато ~12–14 на четырёх-восьми**, а «авто»
(все 20 логических ядер) — не лучше восьми, а в части прогонов и хуже. Полоса
памяти одна на всех (урок 5.2): после четырёх потоков добавка тонет в конкуренции
за неё и в троттлинге.

Практический вывод: не оставляйте дефолт не глядя. Начните с 4–8 потоков или числа
физических ядер — и замерьте на своей машине, это две минуты.


## Киллер-фича: грамматики GBNF

Весь курс мы добивались от модели формата уговорами: «Ответь только JSON»,
few-shot (урок 1.7), Pydantic + retry с текстом ошибки (урок 1.2), «умная»
вырезалка `{...}` в чекерах (урок 5.2). Это работало — в 90–99% случаев.

Грамматика решает задачу иначе: **на каждом шаге генерации сэмплер видит только
те токены, которые допустимы грамматикой**. Модель физически не может вывести
ничего вне формата — не «обещает», а не может. Это гарантия уровня декодера,
как CHECK-ограничение в БД против «пожалуйста, пишите в это поле только числа».

GBNF — это BNF-нотация llama.cpp: `root` — стартовое правило, `|` — альтернативы,
`"..."` — литералы. Простейший случай — жёсткий enum для классификации:


In [ ]:
if LLM:
    sentiment_grammar = LlamaGrammar.from_string(
        'root ::= "positive" | "negative" | "neutral"', verbose=False)
    answer = chat(
        [{"role": "user", "content":
          "Определи тональность отзыва: «Ждал курьера два часа, всё остыло». "
          "Ответь одним словом."}],
        max_tokens=8,
        grammar=sentiment_grammar,
    )
    print(repr(answer))
else:
    print("Пропущено")

Ответ — ровно одно слово из трёх разрешённых, без точки, без «Конечно, вот ответ:».
Сравните с уроком 5.2, где base-модель на этой же задаче выдавала бесконечный поток
мусора: грамматика сделала бы невозможным даже его.

### JSON по схеме — та же грамматика, но собранная за вас

Для JSON не нужно писать GBNF руками: `response_format` с JSON-схемой компилируется
в грамматику автоматически. Это тот же контракт, что Pydantic в уроке 1.2, —
но нарушить его модель не может:


In [ ]:
if LLM:
    answer = chat(
        [{"role": "user", "content": "Извлеки из текста имя и возраст: «Марии 34 года»."}],
        max_tokens=60,
        response_format={
            "type": "json_object",
            "schema": {
                "type": "object",
                "properties": {"name": {"type": "string"}, "age": {"type": "integer"}},
                "required": ["name", "age"],
            },
        },
    )
    print(repr(answer))
    print("json.loads без всяких вырезалок:", json.loads(answer))
else:
    print("Пропущено")

`json.loads` напрямую, без regex-вырезалок и retry. Но держите в голове три предела:

- **Форма ≠ содержание.** Грамматика гарантирует валидный JSON с нужными ключами,
  но не мешает модели положить в них чушь. Проверки смысла (уроки 3.3–3.4) остаются.
- **Цена.** Сложные грамматики замедляют генерацию: на каждый токен сэмплер
  фильтрует словарь. Для enum и небольших схем это незаметно, для огромных схем —
  измеряйте.
- **Грамматика умеет запрещать, а не подсказывать.** Если промпт не подводит модель
  к разрешённому слову, маска заставит её взять «наименее плохой» допустимый токен —
  и он может оказаться одинаковой чушью для любых входов. Живьём — в упражнении 1.

В Ollama, кстати, есть родственник — `format: "json"` и structured outputs
в новых версиях: под капотом тот же механизм llama.cpp. Но полный доступ
к произвольным GBNF-грамматикам даёт именно прямой inference.


## Стриминг

Как и в OpenAI-совместимых API (урок 1.4), `stream=True` превращает ответ
в итератор чанков — только здесь это не SSE по сети, а обычный Python-генератор
в вашем процессе:


In [ ]:
if LLM:
    stream = LLM.create_chat_completion(
        messages=[{"role": "user", "content": "Сосчитай от 1 до 5 через запятую."}],
        max_tokens=30, temperature=0, stream=True,
    )
    pieces = []
    for part in stream:
        delta = part["choices"][0]["delta"]
        if "content" in delta:
            pieces.append(delta["content"])
    print(f"чанков: {len(pieces)}, текст: {''.join(pieces)}")
else:
    print("Пропущено")

## Итоговая таблица: Ollama или llama-cpp-python?

| Критерий | Ollama | llama-cpp-python |
|---|---|---|
| Несколько клиентов/сервисов | ✅ демон, очередь, 503 | ❌ городить самим |
| Управление моделями (pull, версии, теги) | ✅ | ❌ файлы руками |
| keep_alive, автопрогрузка/выгрузка | ✅ | ❌ модель живёт с процессом |
| Встроить в CLI/desktop без зависимостей | ❌ тащить демона | ✅ pip install |
| Грамматики GBNF, logit bias, свой сэмплер | ❌/частично (format: json) | ✅ полный доступ |
| Изоляция от чужих обновлений | ❌ демон общий (история из 5.3!) | ✅ версия зафиксирована в venv |
| Простота установки | ✅ инсталлер | ⚠️ колесо или сборка |

Правило выбора: **сервис для нескольких потребителей — Ollama; встраивание
в одно приложение или жёсткие гарантии формата — llama-cpp-python.**
Гибрид тоже честен: сервис на Ollama, а рядом — маленькая утилита с грамматикой.

## ⚠️ Частые ошибки

1. **Оставить `n_ctx` по умолчанию.** Дефолт — 512 токенов: длинный промпт молча
   обрежется, и модель «забудет» начало. Та же тихая обрезка, что в уроке 1.5, —
   только теперь вы сами отвечаете за размер контекста.
2. **Звать `LLM("текст...")` для instruct-модели.** Прямой вызов — это сырое
   продолжение текста без шаблона чата (та самая base-ловушка из уроков 1.6 и 5.2).
   Для диалога — `create_chat_completion`, шаблон применится из GGUF.
3. **Две копии модели в RAM.** Ollama держит свою (keep_alive!), ваш процесс — свою.
   На ноутбуке с 8 ГБ это уже беда. Работаете с llama-cpp-python — выгрузите
   модель из Ollama (`keep_alive: 0`, урок 5.3).
4. **Верить, что грамматика решает всё.** Она гарантирует форму, не смысл:
   валидный JSON с выдуманным возрастом пройдёт. Проверки фактов (модуль 3) остаются.
5. **Строить продакшен на блобах Ollama.** `ollama rm` — и файла нет. Блоб — удобный
   трюк для экспериментов; в приложение кладите собственный GGUF.
6. **Грамматика без согласованного промпта.** Маска не добавляет смысла: не попросили
   «ответь ровно одним словом» — и модель, собиравшаяся начать с преамбулы, выберет
   один и тот же разрешённый токен для всех входов (см. решение упражнения 1).

## Упражнения

1. **Грамматика приоритетов.** Составьте грамматику для `low | medium | high`
   и прогоните через неё три тикета: «прод лежит, теряем деньги», «поправьте
   опечатку в футере», «не работает экспорт у одного клиента». Совпало ли с вашей
   интуицией? Если модель отвечает странно — не спешите её винить: в решении
   вас ждёт ловушка, ради которой это упражнение и существует.
2. **Схема для заявки.** Повторите задачу `json-struct-2` из бенчмарка урока 5.2
   (item/quantity/city), но через `response_format` со схемой. Сравните с тем,
   что там делал «мягкий» чекер.
3. **Честная дуэль с Ollama.** Замерьте генерацию на 100+ токенов у llama-cpp-python
   и у Ollama в CPU-режиме (`num_gpu: 0`) на **уникальных** промптах (урок 5.2
   объяснил, почему). Какой движок быстрее у вас и на сколько?

Решения — ниже.


## Мини-квиз

<details><summary><b>1. Чем принципиально грамматика GBNF отличается от «Ответь только JSON» в промпте?</b></summary>

Промпт — это просьба: модель следует ей с высокой вероятностью, но может нарушить.
Грамматика работает на уровне сэмплера: недопустимые токены просто исключаются
из выбора на каждом шаге. Формат гарантирован конструктивно, а не статистически.
</details>

<details><summary><b>2. Вы загрузили Llama(model_path=...) без параметров и подали промпт на 2000 токенов. Что будет?</b></summary>

Дефолтный `n_ctx` = 512: промпт молча обрежется (или упадёт с ошибкой — зависит
от версии), модель увидит только часть. Всегда задавайте `n_ctx` явно под свою
задачу — и помните, что KV-кеш растёт вместе с ним (урок 5.2).
</details>

<details><summary><b>3. Почему при работе с llama-cpp-python на слабой машине стоит сначала выгрузить модель из Ollama?</b></summary>

Иначе в RAM живут **две копии** весов: одна у runner'а Ollama (её держит keep_alive),
вторая — в вашем процессе. Для 3B это ~4 ГБ вместо двух, для 8B — ~10. Выгрузка —
запрос с `keep_alive: 0` (урок 5.3).
</details>

<details><summary><b>4. Откуда create_chat_completion знает, как оформить диалог для конкретной модели?</b></summary>

Шаблон чата хранится в метаданных GGUF-файла (вместе с токенизатором и параметрами
архитектуры — «всё в одном файле», урок 5.2). В уроке 1.6 мы собирали такую разметку
руками — теперь видно, откуда библиотеки её берут.
</details>

<details><summary><b>5. Сервису с 20 внутренними клиентами нужен доступ к локальной LLM. Ollama или llama-cpp-python?</b></summary>

Ollama: очередь, параллелизм, 503 при перегрузке, keep_alive и управление моделями
уже встроены (урок 5.3). llama-cpp-python пришлось бы оборачивать в свой сервер
и переизобретать всё это. Прямой inference — для встраивания в одно приложение
и для жёстких гарантий формата.
</details>


## Итоги

- llama-cpp-python — тот же движок llama.cpp, что под Ollama, но в вашем процессе:
  PostgreSQL против SQLite. Сервис для многих — Ollama; встраивание и контроль — прямой inference.
- GGUF можно не скачивать заново: блоб Ollama — это готовый GGUF (manifest → digest → файл).
  Для продакшена — собственный файл.
- `n_ctx` по умолчанию 512 — задавайте явно. Потоки: плато на 4–8, «авто» не обязан быть
  лучшим. Скорость готового колеса сопоставима с Ollama (у меня 10–15 против 15–17 ток/с;
  троттлинг влияет сильнее сборки) — замеряйте свою связку.
- Грамматики GBNF и JSON-схемы гарантируют формат на уровне сэмплера — это сильнее
  любого промпта и retry. Но форма ≠ смысл, и промпт обязан подводить модель
  к разрешённому ответу — иначе маска выберет чушь.
- Шаблон чата берётся из метаданных GGUF — `create_chat_completion` вместо ручного ChatML.

**Дальше**: урок 5.5 — мониторинг локальной LLM: метрики, деградация, алерты.

## Полезные ссылки

- llama-cpp-python: <https://llama-cpp-python.readthedocs.io>
- Синтаксис GBNF: <https://github.com/ggml-org/llama.cpp/blob/master/grammars/README.md>
- Готовые GGUF: <https://huggingface.co/bartowski>
- llama.cpp: <https://github.com/ggml-org/llama.cpp>


---

## Решения упражнений

### Упражнение 1: грамматика приоритетов


In [ ]:
if LLM:
    priority_grammar = LlamaGrammar.from_string(
        'root ::= "low" | "medium" | "high"', verbose=False)
    tickets = [
        "Прод лежит, клиенты не могут оплатить, теряем деньги каждую минуту!",
        "Поправьте, пожалуйста, опечатку в футере сайта.",
        "У одного из клиентов не работает экспорт отчёта в Excel.",
    ]

    def classify_all(template: str) -> None:
        for ticket in tickets:
            out = LLM.create_chat_completion(
                messages=[{"role": "user", "content": template.format(t=ticket)}],
                max_tokens=4, temperature=0, seed=42, grammar=priority_grammar)
            print(f"  {out['choices'][0]['message']['content']:<8} <- {ticket[:55]}")

    print("Наивный промпт (грамматика есть, подводки нет):")
    classify_all("Определи приоритет тикета: low, medium или high.\n\nТикет: «{t}»")

    print("\nПромпт, согласованный с грамматикой:")
    classify_all("Определи приоритет тикета: low, medium или high. "
                 "Ответь ровно одним словом.\n\nТикет: «{t}»")
else:
    print("Пропущено")

Ловушка, ради которой упражнение существует: с наивным промптом у меня **все три
тикета получили `low`** — включая лежащий прод. Модель собиралась начать ответ
чем-то вроде «Приоритет этого тикета…», грамматика такие токены запретила, и argmax
среди разрешённых оказался одинаковым и бессодержательным для любых входов.

Грамматика — маска на сэмплере, а не источник смысла. **Промпт должен подводить
модель ровно к тому слову, которое грамматика разрешает** («Ответь ровно одним
словом») — тогда у меня выходит high/medium/medium: с этим уже можно работать
(спор о medium для сломанного экспорта — вопрос калибровки формулировок,
а не сломанного формата).


### Упражнение 2: схема для заявки


In [ ]:
if LLM:
    answer = chat(
        [{"role": "user", "content":
          "Извлеки из заявки данные. Заявка: «Пришлите, пожалуйста, "
          "3 механические клавиатуры в наш офис в Казани»."}],
        max_tokens=80,
        response_format={
            "type": "json_object",
            "schema": {
                "type": "object",
                "properties": {
                    "item": {"type": "string"},
                    "quantity": {"type": "integer"},
                    "city": {"type": "string"},
                },
                "required": ["item", "quantity", "city"],
            },
        },
    )
    print(json.loads(answer))
    # В уроке 5.2 чекер вырезал первый {...} из ответа регуляркой — «на всякий случай».
    # Здесь вырезать нечего: кроме JSON, модель не могла сгенерировать ничего.
else:
    print("Пропущено")

### Упражнение 3: честная дуэль с Ollama

Методика важнее цифр: **уникальные** промпты (кеш префикса из урока 5.2 не должен
подыгрывать никому), одинаковый `max_tokens`, обоим — чистый CPU. У Ollama это
`options: {"num_gpu": 0}`, у llama-cpp-python CPU и так по умолчанию.


In [ ]:
import httpx

OLLAMA_URL = "http://127.0.0.1:11434"
NS = 1_000_000_000

if LLM:
    prompt_a = "Вопрос 101. Перечисли 20 профессий будущего через запятую: "
    prompt_b = "Вопрос 202. Перечисли 20 видов транспорта через запятую: "

    # --- llama-cpp-python ---
    t0 = time.perf_counter()
    out = LLM.create_completion(prompt_a, max_tokens=100, temperature=0, seed=42)
    lcp_tps = out["usage"]["completion_tokens"] / (time.perf_counter() - t0)

    # --- Ollama, чистый CPU, уникальный промпт ---
    try:
        d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": "qwen2.5:3b", "prompt": prompt_b, "stream": False,
            "options": {"temperature": 0, "seed": 42, "num_predict": 100, "num_gpu": 0},
        }, timeout=600).raise_for_status().json()
        ollama_tps = d["eval_count"] / (d["eval_duration"] / NS)
        # вернуть дефолтный (не-CPU) runner следующему пользователю ноутбука
        httpx.post(f"{OLLAMA_URL}/api/generate",
                   json={"model": "qwen2.5:3b", "keep_alive": 0}, timeout=60)
        print(f"llama-cpp-python: {lcp_tps:.1f} ток/с")
        print(f"Ollama (CPU):     {ollama_tps:.1f} ток/с")
        print(f"разница:          x{ollama_tps / lcp_tps:.2f} в пользу"
              f" {'Ollama' if ollama_tps > lcp_tps else 'llama-cpp-python'}")
    except httpx.HTTPError as exc:
        print(f"llama-cpp-python: {lcp_tps:.1f} ток/с; Ollama недоступна ({exc.__class__.__name__})")
else:
    print("Пропущено")